In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Diabetes Logistic Regression") \
    .getOrCreate()

df = spark.read.csv("hdfs://localhost:9000/user/yash1122/DataSet/Diabetes/diabetes_binary_5050split_health_indicators_BRFSS2015.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)


root
 |-- Diabetes_binary: double (nullable = true)
 |-- HighBP: double (nullable = true)
 |-- HighChol: double (nullable = true)
 |-- CholCheck: double (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Smoker: double (nullable = true)
 |-- Stroke: double (nullable = true)
 |-- HeartDiseaseorAttack: double (nullable = true)
 |-- PhysActivity: double (nullable = true)
 |-- Fruits: double (nullable = true)
 |-- Veggies: double (nullable = true)
 |-- HvyAlcoholConsump: double (nullable = true)
 |-- AnyHealthcare: double (nullable = true)
 |-- NoDocbcCost: double (nullable = true)
 |-- GenHlth: double (nullable = true)
 |-- MentHlth: double (nullable = true)
 |-- PhysHlth: double (nullable = true)
 |-- DiffWalk: double (nullable = true)
 |-- Sex: double (nullable = true)
 |-- Age: double (nullable = true)
 |-- Education: double (nullable = true)
 |-- Income: double (nullable = true)

+---------------+------+--------+---------+----+------+------+--------------------+------------+---

In [11]:
df.columns

['Diabetes_binary',
 'HighBP',
 'HighChol',
 'CholCheck',
 'BMI',
 'Smoker',
 'Stroke',
 'HeartDiseaseorAttack',
 'PhysActivity',
 'Fruits',
 'Veggies',
 'HvyAlcoholConsump',
 'AnyHealthcare',
 'NoDocbcCost',
 'GenHlth',
 'MentHlth',
 'PhysHlth',
 'DiffWalk',
 'Sex',
 'Age',
 'Education',
 'Income']

In [5]:
from pyspark.sql.functions import col, isnan, when, count

df.select([count(when(col(c).isNull() | isnan(c), c)).alias(c) for c in df.columns]).show()



+---------------+------+--------+---------+---+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-------+--------+--------+--------+---+---+---------+------+
|Diabetes_binary|HighBP|HighChol|CholCheck|BMI|Smoker|Stroke|HeartDiseaseorAttack|PhysActivity|Fruits|Veggies|HvyAlcoholConsump|AnyHealthcare|NoDocbcCost|GenHlth|MentHlth|PhysHlth|DiffWalk|Sex|Age|Education|Income|
+---------------+------+--------+---------+---+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-------+--------+--------+--------+---+---+---------+------+
|              0|     0|       0|        0|  0|     0|     0|                   0|           0|     0|      0|                0|            0|          0|      0|       0|       0|       0|  0|  0|        0|     0|
+---------------+------+--------+---------+---+------+------+--------------------+------------+------+-------+-----------------+------------

In [15]:
from pyspark.ml.feature import VectorAssembler


input_features = df.columns[1:]

assembler = VectorAssembler(inputCols=input_features, outputCol="features")

assembled_df = assembler.transform(df).select("features", "Diabetes_binary")


In [28]:
from pyspark.ml.classification import LogisticRegression

train_data, test_data = assembled_df.randomSplit([0.8, 0.2], seed=42)

lr = LogisticRegression(labelCol="Diabetes_binary", featuresCol="features")

lr_model = lr.fit(train_data)

results = lr_model.evaluate(test_data)

print("Accuracy:", results.accuracy)


Accuracy: 0.744240614334471


In [23]:
# Make predictions
predictions = lr_model.transform(test_data)
predictions.select("Diabetes_binary", "prediction").show(100, truncate=False)


+---------------+----------+
|Diabetes_binary|prediction|
+---------------+----------+
|1.0            |1.0       |
|1.0            |1.0       |
|0.0            |1.0       |
|1.0            |1.0       |
|0.0            |1.0       |
|0.0            |1.0       |
|0.0            |1.0       |
|0.0            |1.0       |
|0.0            |1.0       |
|1.0            |1.0       |
|0.0            |1.0       |
|0.0            |1.0       |
|0.0            |1.0       |
|1.0            |1.0       |
|0.0            |1.0       |
|1.0            |1.0       |
|0.0            |1.0       |
|1.0            |1.0       |
|1.0            |1.0       |
|0.0            |1.0       |
|0.0            |1.0       |
|1.0            |0.0       |
|0.0            |1.0       |
|0.0            |0.0       |
|0.0            |1.0       |
|0.0            |1.0       |
|0.0            |1.0       |
|0.0            |1.0       |
|0.0            |1.0       |
|0.0            |1.0       |
|1.0            |1.0       |
|0.0          